# Sweep Configuration Comparison

Compares all **12 sweep configurations** (2 frequencies × 6 TX powers, SF12 / BW125 / CR4/8) from `runs_csv/`.

Every plot includes all 12 configurations. Color encoding is consistent throughout:
- **433.1 MHz → blue**, **446.1 MHz → orange** (solid vs. dashed lines, circle vs. square markers)
- **Darker shade = higher TX power** (4 → 22 dBm)

Per-link views use small multiples (one panel per sender or per link) instead of piling all 56 links into one chart.

Failed receptions (`ok_match == 0`) still carry an RSSI value but have `snr == 0`, so they are **excluded from RSSI/SNR/latency statistics** and only count toward PDR.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

DATA_DIR = Path("runs_csv")
if not DATA_DIR.exists():  # allow running from inside SOFTWARE/ or runs_csv/
    for cand in [Path("../runs_csv"), Path(".")]:
        if (cand / "runs_manifest.csv").exists():
            DATA_DIR = cand
            break

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2,
    "font.size": 10,
})

## Load data

In [ ]:
files = sorted(DATA_DIR.glob("results_*.csv"))
raw = pd.concat([pd.read_csv(f).assign(csv_file=f.name) for f in files], ignore_index=True)
raw["pc_ts"] = pd.to_datetime(raw["pc_ts"])
raw["link"] = raw["sender_id"].astype(str) + "→" + raw["receiver_id"].astype(str)

FREQS = sorted(raw["freq_mhz"].unique())
POWERS = sorted(raw["tx_power"].unique())
CONFIGS = [(f, p) for f in FREQS for p in POWERS]           # fixed order: freq, then power
NODES = sorted(set(raw["sender_id"]) | set(raw["receiver_id"]))

def cfg_label(f, p):
    return f"{f:g} MHz / {p:g} dBm"

raw["config"] = [cfg_label(f, p) for f, p in zip(raw["freq_mhz"], raw["tx_power"])]
CONFIG_LABELS = [cfg_label(f, p) for f, p in CONFIGS]

ok = raw[raw["ok_match"] == 1]   # successful receptions only (valid RSSI/SNR/latency)

print(f"{len(files)} runs, {len(raw):,} rows, {raw['link'].nunique()} directed links, nodes {NODES}")
raw.groupby("config", sort=False)["ok_match"].agg(rows="size", pdr_pct=lambda s: 100 * s.mean()).reindex(CONFIG_LABELS).round(2)

## Color / style encoding

In [ ]:
FREQ_CMAP = {FREQS[0]: mpl.colormaps["Blues"], FREQS[1]: mpl.colormaps["Oranges"]}
FREQ_LS = {FREQS[0]: "-", FREQS[1]: "--"}
FREQ_MARKER = {FREQS[0]: "o", FREQS[1]: "s"}

def cfg_color(f, p):
    # light -> dark with power; keep away from the washed-out low end of the ramp
    t = 0.35 + 0.6 * POWERS.index(p) / (len(POWERS) - 1)
    return FREQ_CMAP[f](t)

COLORS = {cfg_label(f, p): cfg_color(f, p) for f, p in CONFIGS}
FREQ_BASE = {f: FREQ_CMAP[f](0.75) for f in FREQS}

def config_legend(ax=None, fig=None, **kw):
    handles = [Line2D([], [], color=cfg_color(f, p), ls=FREQ_LS[f], marker=FREQ_MARKER[f], ms=6,
                      label=cfg_label(f, p)) for f, p in CONFIGS]
    target = fig if fig is not None else ax
    kw.setdefault("fontsize", 8)
    return target.legend(handles=handles, ncol=2, title="Configuration", **kw)

def freq_legend(ax, **kw):
    handles = [Line2D([], [], color=FREQ_BASE[f], ls=FREQ_LS[f], marker=FREQ_MARKER[f], label=f"{f:g} MHz") for f in FREQS]
    return ax.legend(handles=handles, **kw)

fig, ax = plt.subplots(figsize=(8, 1.6))
for i, (f, p) in enumerate(CONFIGS):
    ax.bar(i, 1, color=cfg_color(f, p), edgecolor="white", linewidth=2)
ax.set_xticks(range(len(CONFIGS)), [cfg_label(f, p).replace(" / ", "\n") for f, p in CONFIGS], fontsize=7)
ax.set_yticks([]); ax.grid(False); ax.set_title("Configuration palette")
plt.show()

## 1. Headline metrics vs. TX power
Each panel shows all 12 configs: x = TX power, one line per frequency.

In [ ]:
run = (raw.groupby(["freq_mhz", "tx_power"])
         .agg(pdr_pct=("ok_match", lambda s: 100 * s.mean()))
         .join(ok.groupby(["freq_mhz", "tx_power"]).agg(
             rssi_mean=("rssi", "mean"), rssi_p10=("rssi", lambda s: s.quantile(0.10)),
             snr_mean=("snr", "mean"), snr_p10=("snr", lambda s: s.quantile(0.10)),
             e2e_median=("e2e_ms", "median")))
         .reset_index())

metrics = [("pdr_pct", "Overall PDR (%)"), ("rssi_mean", "Mean RSSI (dBm)"), ("rssi_p10", "RSSI 10th pct (dBm)"),
           ("snr_mean", "Mean SNR (dB)"), ("snr_p10", "SNR 10th pct (dB)"), ("e2e_median", "Median E2E latency (ms)")]

fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, (col, title) in zip(axes.flat, metrics):
    for f in FREQS:
        d = run[run.freq_mhz == f].sort_values("tx_power")
        ax.plot(d.tx_power, d[col], color=FREQ_BASE[f], ls=FREQ_LS[f], alpha=0.6)
        ax.scatter(d.tx_power, d[col], c=[cfg_color(f, p) for p in d.tx_power], marker=FREQ_MARKER[f],
                   s=60, edgecolor="white", linewidth=1.5, zorder=3)
    ax.set_title(title); ax.set_xlabel("TX power (dBm)"); ax.set_xticks(POWERS)
freq_legend(axes[0, 0], loc="lower left")
fig.suptitle("Per-configuration summary", fontsize=13)
fig.tight_layout(); plt.show()
run.round(2)

## 2. RSSI and SNR distributions per configuration

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, col, ylabel in [(axes[0], "rssi", "RSSI (dBm)"), (axes[1], "snr", "SNR (dB)")]:
    data = [ok.loc[ok.config == c, col].values for c in CONFIG_LABELS]
    parts = ax.violinplot(data, positions=range(len(CONFIG_LABELS)), showextrema=False, widths=0.85)
    for body, c in zip(parts["bodies"], CONFIG_LABELS):
        body.set_facecolor(COLORS[c]); body.set_edgecolor("none"); body.set_alpha(0.55)
    ax.boxplot(data, positions=range(len(CONFIG_LABELS)), widths=0.18, showfliers=False,
               medianprops=dict(color="black"), boxprops=dict(color="#444"), whiskerprops=dict(color="#444"),
               capprops=dict(color="#444"))
    ax.set_ylabel(ylabel)
    ax.axvline(len(POWERS) - 0.5, color="#888", lw=1, ls=":")
axes[1].set_xticks(range(len(CONFIG_LABELS)), [c.replace(" / ", "\n") for c in CONFIG_LABELS], fontsize=8)
axes[0].set_title("Distribution over all successful packets (violin = density, box = IQR)")
fig.tight_layout(); plt.show()

## 3. Cumulative distributions (ECDF)
The tail on the left is what matters for link margin: how often is a packet received with weak RSSI / low SNR?

In [ ]:
def ecdf(x):
    x = np.sort(x); return x, np.arange(1, len(x) + 1) / len(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, xlabel in [(axes[0], "rssi", "RSSI (dBm)"), (axes[1], "snr", "SNR (dB)")]:
    for f, p in CONFIGS:
        x, y = ecdf(ok.loc[(ok.freq_mhz == f) & (ok.tx_power == p), col].values)
        ax.step(x, y, where="post", color=cfg_color(f, p), ls=FREQ_LS[f], lw=1.8)
    ax.set_xlabel(xlabel); ax.set_ylabel("Fraction of packets ≤ x")
config_legend(axes[1], loc="upper left")
fig.suptitle("ECDF per configuration", fontsize=13)
fig.tight_layout(); plt.show()

## 4. Stability over the run
Rolling mean over rounds (the runs happened at different times of day, so x is the round index within each run). Shows drift, interference bursts, or temperature effects.

In [ ]:
ROLL = 10  # rounds

per_round_ok = (ok.groupby(["freq_mhz", "tx_power", "round_id"]).agg(rssi=("rssi", "mean"), snr=("snr", "mean")))
per_round_pdr = raw.groupby(["freq_mhz", "tx_power", "round_id"])["ok_match"].mean().mul(100).rename("pdr")
per_round = per_round_ok.join(per_round_pdr, how="outer").reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for f, p in CONFIGS:
    d = per_round[(per_round.freq_mhz == f) & (per_round.tx_power == p)].sort_values("round_id")
    idx = np.arange(len(d))
    for ax, col in zip(axes, ["rssi", "snr", "pdr"]):
        ax.plot(idx, d[col].rolling(ROLL, min_periods=1).mean(), color=cfg_color(f, p), ls=FREQ_LS[f], lw=1.6)
axes[0].set_ylabel("RSSI (dBm)"); axes[1].set_ylabel("SNR (dB)"); axes[2].set_ylabel("PDR (%)")
axes[2].set_xlabel("Round index within run")
axes[0].set_title(f"Per-round means, {ROLL}-round rolling average")
config_legend(fig=fig, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout(); plt.show()

## 5. Latency per configuration

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for f, p in CONFIGS:
    x, y = ecdf(ok.loc[(ok.freq_mhz == f) & (ok.tx_power == p), "e2e_ms"].values)
    ax.step(x, y, where="post", color=cfg_color(f, p), ls=FREQ_LS[f], lw=1.6)
ax.set_xlabel("End-to-end latency, successful packets (ms)"); ax.set_ylabel("Fraction ≤ x")
ax.set_title("E2E latency ECDF (steps come from the receiver polling order)")
config_legend(ax, loc="lower right")
fig.tight_layout(); plt.show()

## 6. Per-link statistics
Aggregate each directed link per configuration, then view them in small multiples.

In [ ]:
link = (raw.groupby(["freq_mhz", "tx_power", "sender_id", "receiver_id"])
          .agg(attempts=("ok_match", "size"), pdr=("ok_match", lambda s: 100 * s.mean()))
          .join(ok.groupby(["freq_mhz", "tx_power", "sender_id", "receiver_id"])
                  .agg(rssi=("rssi", "mean"), rssi_std=("rssi", "std"), snr=("snr", "mean"), snr_min=("snr", "min")))
          .reset_index())
link["link"] = link.sender_id.astype(str) + "→" + link.receiver_id.astype(str)
worst = link.sort_values("pdr").head(15)
worst

### 6a. Mean RSSI per link — one panel per sender
x = receiver, 12 markers per receiver (one per configuration).

In [ ]:
def per_sender_panels(col, ylabel, title):
    fig, axes = plt.subplots(2, 4, figsize=(16, 7.5), sharey=True)
    offsets = np.linspace(-0.38, 0.38, len(CONFIGS))
    for ax, s in zip(axes.flat, NODES):
        d = link[link.sender_id == s]
        receivers = [r for r in NODES if r != s]
        for k, (f, p) in enumerate(CONFIGS):
            dd = d[(d.freq_mhz == f) & (d.tx_power == p)].set_index("receiver_id").reindex(receivers)
            ax.scatter(np.arange(len(receivers)) + offsets[k], dd[col], color=cfg_color(f, p),
                       marker=FREQ_MARKER[f], s=22, edgecolor="white", linewidth=0.5, zorder=3)
        ax.set_xticks(range(len(receivers)), receivers)
        for i in range(len(receivers) - 1):
            ax.axvline(i + 0.5, color="#ddd", lw=0.8)
        ax.grid(axis="x", visible=False)
        ax.set_title(f"Sender {s}"); ax.set_xlabel("Receiver")
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel)
    config_legend(fig=fig, loc="center left", bbox_to_anchor=(1.0, 0.5))
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(); plt.show()

per_sender_panels("rssi", "Mean RSSI (dBm)", "Mean RSSI per link, all configurations")

### 6b. Mean SNR per link — one panel per sender

In [ ]:
per_sender_panels("snr", "Mean SNR (dB)", "Mean SNR per link, all configurations")

### 6c. PDR per link — one panel per sender

In [ ]:
per_sender_panels("pdr", "PDR (%)", "PDR per link, all configurations")

### 6d. RSSI response to TX power, link by link
One small panel per directed link (row = sender, column = receiver). Each panel has the 6 powers × 2 frequencies. A healthy link rises roughly 1 dB per dB of TX power until the receiver saturates.

In [ ]:
def link_grid(col, ylabel, title, sharey=True):
    n = len(NODES)
    fig, axes = plt.subplots(n, n, figsize=(2.1 * n, 1.8 * n), sharex=True, sharey=sharey)
    for i, s in enumerate(NODES):
        for j, r in enumerate(NODES):
            ax = axes[i, j]
            if s == r:
                ax.axis("off"); continue
            for f in FREQS:
                d = link[(link.sender_id == s) & (link.receiver_id == r) & (link.freq_mhz == f)].sort_values("tx_power")
                ax.plot(d.tx_power, d[col], color=FREQ_BASE[f], ls=FREQ_LS[f], lw=1.2, alpha=0.7)
                ax.scatter(d.tx_power, d[col], c=[cfg_color(f, p) for p in d.tx_power], marker=FREQ_MARKER[f],
                           s=14, zorder=3)
            ax.set_title(f"{s}→{r}", fontsize=8, pad=2)
            ax.tick_params(labelsize=7)
    for ax in axes[-1, :]:
        ax.set_xlabel("TX dBm", fontsize=8)
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel, fontsize=8)
    freq_legend(axes[0, 0], loc="center", fontsize=9)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(); plt.show()

link_grid("rssi", "RSSI", "Mean RSSI vs TX power per link (row = sender, col = receiver)")

### 6e. SNR response to TX power, link by link

In [ ]:
link_grid("snr", "SNR", "Mean SNR vs TX power per link (row = sender, col = receiver)")

### 6f. PDR response to TX power, link by link
Independent y-axes so small losses are visible; flat panels at 100 % are fine.

In [ ]:
link_grid("pdr", "PDR %", "PDR vs TX power per link (row = sender, col = receiver)", sharey=False)

## 7. Link matrices — one heatmap per configuration
Same color scale across all 12 panels so they can be compared directly.

In [ ]:
def config_heatmaps(col, label, cmap, vmin=None, vmax=None):
    vmin = link[col].min() if vmin is None else vmin
    vmax = link[col].max() if vmax is None else vmax
    fig, axes = plt.subplots(len(FREQS), len(POWERS), figsize=(3 * len(POWERS), 3.2 * len(FREQS)))
    for i, f in enumerate(FREQS):
        for j, p in enumerate(POWERS):
            ax = axes[i, j]
            m = (link[(link.freq_mhz == f) & (link.tx_power == p)]
                 .pivot(index="sender_id", columns="receiver_id", values=col).reindex(index=NODES, columns=NODES))
            im = ax.imshow(m.values, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_xticks(range(len(NODES)), NODES, fontsize=7); ax.set_yticks(range(len(NODES)), NODES, fontsize=7)
            ax.grid(False)
            ax.set_title(cfg_label(f, p), fontsize=9)
            if j == 0: ax.set_ylabel("Sender")
            if i == len(FREQS) - 1: ax.set_xlabel("Receiver")
    fig.colorbar(im, ax=axes, shrink=0.7, label=label)
    plt.show()

config_heatmaps("rssi", "Mean RSSI (dBm)", "viridis")

In [ ]:
config_heatmaps("snr", "Mean SNR (dB)", "viridis")

In [ ]:
config_heatmaps("pdr", "PDR (%)", "magma", vmax=100)

## 8. Frequency comparison: 446.1 − 433.1 MHz per link
Diverging scale, gray = no difference. Positive = 446.1 MHz is better. One panel per TX power.

In [ ]:
def freq_delta(col, label, lim=None):
    piv = link.pivot_table(index=["tx_power", "sender_id", "receiver_id"], columns="freq_mhz", values=col)
    piv["delta"] = piv[FREQS[1]] - piv[FREQS[0]]
    lim = lim or np.nanmax(np.abs(piv["delta"]))
    fig, axes = plt.subplots(1, len(POWERS), figsize=(3 * len(POWERS), 3.4))
    for ax, p in zip(axes, POWERS):
        m = piv.loc[p, "delta"].unstack().reindex(index=NODES, columns=NODES)
        im = ax.imshow(m.values, cmap="RdBu", vmin=-lim, vmax=lim)
        ax.set_xticks(range(len(NODES)), NODES, fontsize=7); ax.set_yticks(range(len(NODES)), NODES, fontsize=7)
        ax.grid(False); ax.set_title(f"{p:g} dBm", fontsize=9); ax.set_xlabel("Receiver")
    axes[0].set_ylabel("Sender")
    fig.colorbar(im, ax=axes, shrink=0.8, label=label)
    plt.show()

freq_delta("rssi", "Δ RSSI (dB)")
freq_delta("snr", "Δ SNR (dB)")
freq_delta("pdr", "Δ PDR (pp)")

## 9. Link reciprocity (A→B vs B→A)
Mean RSSI of each direction for every node pair. Points on the diagonal are symmetric links; one panel per frequency, colored by power.

In [ ]:
pairs = link.merge(link, left_on=["freq_mhz", "tx_power", "sender_id", "receiver_id"],
                   right_on=["freq_mhz", "tx_power", "receiver_id", "sender_id"], suffixes=("_ab", "_ba"))
pairs = pairs[pairs.sender_id_ab < pairs.receiver_id_ab]

fig, axes = plt.subplots(1, len(FREQS), figsize=(13, 6), sharex=True, sharey=True)
lo, hi = pairs[["rssi_ab", "rssi_ba"]].min().min() - 2, pairs[["rssi_ab", "rssi_ba"]].max().max() + 2
for ax, f in zip(axes, FREQS):
    ax.plot([lo, hi], [lo, hi], color="#999", lw=1, ls=":")
    for p in POWERS:
        d = pairs[(pairs.freq_mhz == f) & (pairs.tx_power == p)]
        ax.scatter(d.rssi_ab, d.rssi_ba, color=cfg_color(f, p), marker=FREQ_MARKER[f], s=40,
                   edgecolor="white", linewidth=0.8, label=f"{p:g} dBm")
    ax.set_title(f"{f:g} MHz"); ax.set_xlabel("RSSI A→B (dBm, A<B)"); ax.set_aspect("equal")
    ax.legend(title="TX power", fontsize=8)
axes[0].set_ylabel("RSSI B→A (dBm)")
fig.tight_layout(); plt.show()

## 10. Link ranking across configurations
Each line is a configuration; x = links sorted by their mean RSSI across all configs. Shows whether a config shifts all links uniformly or reorders them.

In [ ]:
order = link.groupby("link")["rssi"].mean().sort_values().index
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
for f, p in CONFIGS:
    d = link[(link.freq_mhz == f) & (link.tx_power == p)].set_index("link").reindex(order)
    axes[0].plot(range(len(order)), d.rssi, color=cfg_color(f, p), ls=FREQ_LS[f], lw=1.4, marker=FREQ_MARKER[f], ms=3)
    axes[1].plot(range(len(order)), d.snr, color=cfg_color(f, p), ls=FREQ_LS[f], lw=1.4, marker=FREQ_MARKER[f], ms=3)
axes[0].set_ylabel("Mean RSSI (dBm)"); axes[1].set_ylabel("Mean SNR (dB)")
axes[1].set_xticks(range(len(order)), order, rotation=90, fontsize=7)
axes[1].set_xlabel("Link (sorted weakest → strongest by mean RSSI)")
config_legend(fig=fig, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout(); plt.show()